# EGT Human Studies: Observational & Interventional

**Observational:** cohort, cross-sectional, case-control studies.
**Interventional:** pure ergothioneine RCTs only (mushroom-based excluded due to coincident bioactives).

Data: 1,158 PubMed records filtered by design type and endpoint.

In [1]:
import pandas as pd, numpy as np, re
import matplotlib, matplotlib.pyplot as plt
matplotlib.use('Agg')

df = pd.read_csv('egt_pubmed_titles.csv')
df['year'] = pd.to_numeric(df['year'], errors='coerce')

# ─── OBSERVATIONAL ───
obs_markers = [
    r'cohort', r'population.based', r'prospective', r'cross.sectional',
    r'case.control', r'NHANES', r'Malm', r'Bogalusa', r'Rotterdam',
    r'Centenarian', r'mendelian.randomiz', r'epidemiolog', r'observational',
    r'longitudinal', r'follow.up', r'dietary.*intake',
    r'mushroom.*(consumption|intake)', r'plasma.*biomarker',
    r'serum.*metaboli', r'blood.*metaboli', r'plasma.*metaboli',
    r'diet.*association', r'food.*frequency',
    r'associated with (reduced|lower|higher|decreased)',
    r'predicts (cognitive|functional|mortality)',
    r'frailty markers.*blood', r'plasma.*(mortality|survival|longevity)',
]

obs_exclude = [
    r'randomized', r'placebo.controlled', r'double.blind', r'clinical.trial',
    r'supplementation.*trial', r'pilot.*study.*(patient|subject)',
    r'open.label', r'in vitro', r'cell line', r'mouse', r'rat', r'mice',
    r'zebrafish', r'biosynthesis', r'fermentation', r'e.?coli',
    r'corynebacterium', r'enzyme.*character', r'crystal structure', r'review',
    r'protocol$', r'^protocol',
]

obs_endpoints = {
    'Mortality / Survival': [r'mortality', r'survival', r'all.cause', r'death'],
    'Cardiovascular': [r'heart', r'cardio', r'vascular', r'endothelial', r'stroke', r'coronary', r'cvd'],
    'Cognitive / Neurodegeneration': [r'cogniti', r'memory', r'dementia', r'alzheimer', r'parkinson', r'brain.*health', r'neurodegen', r'mental'],
    'Physical Function / Frailty': [r'grip', r'gait', r'sarcopenia', r'frailty', r'physical.*(function|performance)', r'walking', r'mobility'],
    'Metabolic / Diabetes': [r'diabetes', r'insulin', r'glucose', r'obes', r'metabolic.syndrome', r'lipid', r'cholesterol'],
    'Inflammation / Immune': [r'inflamm', r'cytokine', r'immune', r'crp'],
    'Aging / Senescence': [r'senescence', r'aging', r'ageing', r'longevity', r'telomere', r'geroprotector'],
    'Nutrition / Dietary': [r'diet', r'nutrition', r'intake', r'consumption', r'micronutrient'],
}

def is_obs(title):
    tl = str(title).lower()
    if any(re.search(p, tl) for p in obs_exclude): return False
    return any(re.search(p, tl) for p in obs_markers)

obs_df = df[df['title'].apply(is_obs) & (df['year'] >= 2017)].copy()

def classify(title, ep_dict):
    tl = str(title).lower()
    return [ep for ep, pats in ep_dict.items() if any(re.search(p, tl) for p in pats)]

obs_df['endpoints'] = obs_df['title'].apply(lambda t: classify(t, obs_endpoints))

years = sorted(y for y in obs_df['year'].unique() if y >= 2017)
obs_table = pd.DataFrame(0, index=list(obs_endpoints.keys()), columns=years)
for _, row in obs_df.iterrows():
    for ep in row['endpoints']:
        obs_table.at[ep, int(row['year'])] += 1
obs_table['Total'] = obs_table.sum(axis=1)
obs_table = obs_table[obs_table['Total'] >= 1]

print('OBSERVATIONAL STUDIES')
print('=' * 80)
print(obs_table.to_string())
print(f'\nStudies: {len(obs_df)}, hits: {obs_table["Total"].sum()}')


OBSERVATIONAL STUDIES
                               2020  2021  2022  2023  2024  2025  2026  Total
Mortality / Survival              1     2     0     0     0     0     0      3
Cardiovascular                    2     0     0     0     0     0     0      2
Cognitive / Neurodegeneration     1     1     2     0     2     0     2      8
Physical Function / Frailty       2     0     0     0     0     0     0      2
Aging / Senescence                0     0     0     0     2     0     0      2
Nutrition / Dietary               0     3     1     0     0     0     2      6

Studies: 21, hits: 23


## Interventional: Pure EGT RCTs Only

Mushroom-based studies excluded (coincident bioactives confound EGT attribution).

In [2]:
# Pure EGT RCT markers — must mention ergothioneine directly + trial design
rct_markers = [
    r'ergothioneine.*(randomized|randomised|placebo|double.blind|pilot|trial)',
    r'(randomized|randomised|double.blind|placebo.controlled).*ergothioneine',
    r'ergothioneine.*supplementation', 
]

# Exclude mushroom-based studies and non-human
rct_exclude = [
    r'mushroom', r'oyster', r'pleurotus', r'hericium', r'ganoderma',
    r'cordyceps', r'agaricus', r'lentinula', r'fungal', r'mycelia',
    r'in vitro', r'mouse', r'rat', r'mice', r'review',
]

rct_endpoints = {
    'Cognitive / Brain': [r'cogniti', r'memory', r'dementia', r'alzheimer', r'brain', r'mental'],
    'Metabolic / Diabetes': [r'diabetes', r'insulin', r'glucose', r'metabolic', r'obes', r'lipid'],
    'Sleep / Fatigue': [r'sleep', r'fatigue', r'psqi', r'insomnia'],
    'Inflammation / Immune': [r'inflamm', r'cytokine', r'immune', r'crp'],
    'Physical Function': [r'exercise', r'performance', r'strength', r'ergogenic'],
    'Safety / Tolerability': [r'safety', r'tolerab', r'adverse'],
    'Ovarian / Reproductive': [r'ovarian', r'fertility', r'menopaus', r'hormon'],
}

def is_pure_egt_rct(title):
    tl = str(title).lower()
    if any(re.search(p, tl) for p in rct_exclude): return False
    return any(re.search(p, tl) for p in rct_markers)

rct_df = df[df['title'].apply(is_pure_egt_rct) & (df['year'] >= 2017)].copy()
rct_df['endpoints'] = rct_df['title'].apply(lambda t: classify(t, rct_endpoints))

rct_table = pd.DataFrame(0, index=list(rct_endpoints.keys()), columns=years)
for _, row in rct_df.iterrows():
    for ep in row['endpoints']:
        rct_table.at[ep, int(row['year'])] += 1
rct_table['Total'] = rct_table.sum(axis=1)
rct_table = rct_table[rct_table['Total'] >= 1]

print('PURE EGT RCTs')
print('=' * 80)
print(rct_table.to_string())
print(f'\nStudies: {len(rct_df)}, hits: {rct_table["Total"].sum()}')
print('\nStudies found:')
for _, row in rct_df.iterrows():
    print(f'  PMID {row.pmid} [{int(row.year)}] {str(row.title)[:130]}')


PURE EGT RCTs
                        2020  2021  2022  2023  2024  2025  2026  Total
Cognitive / Brain          0     0     0     0     1     0     0      1
Metabolic / Diabetes       0     1     0     0     0     0     0      1
Ovarian / Reproductive     0     0     0     0     0     0     1      1

Studies: 8, hits: 3

Studies found:
  PMID 42310611 [2026] Oral L-ergothioneine supplementation in women with suboptimal ovarian reserve: a single-center, open-label, self-controlled pilot 
  PMID 40922697 [2025] Ergothioneine as a promising natural antioxidant: bioactivities, therapeutic potential, and industrial applications.
  PMID 40768667 [2025] Ergothioneine supplementation improves pup phenotype and survival in a murine model of spinal muscular atrophy.
  PMID 39544014 [2024] Investigating the efficacy of ergothioneine to delay cognitive decline in mild cognitively impaired subjects: A pilot study.
  PMID 39328438 [2024] Effects of ergothioneine supplementation on meiotic competenc

## Side-by-Side Tables (rendered image)

In [3]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 6))
fig.patch.set_facecolor('#1a1a1a')

for ax, table, title in [(ax1, obs_table, 'Observational (Epidemiological)'), (ax2, rct_table, 'Interventional (Pure EGT RCTs)')]:
    ax.axis('off')
    ax.set_facecolor('#1a1a1a')
    col_labels = [str(c) if str(c) != 'Total' else '∑' for c in table.columns]
    cell_text = [[str(int(v)) if v > 0 else '' for v in row] for _, row in table.iterrows()]
    
    tbl = ax.table(cellText=cell_text, rowLabels=list(table.index), colLabels=col_labels,
                   cellLoc='center', rowLoc='center',
                   colWidths=[0.05]*len(col_labels),
                   bbox=[0.0, 0.1, 0.95, 0.85])
    tbl.auto_set_font_size(False)
    tbl.set_fontsize(7)
    
    for i in range(len(table)):
        for j in range(len(table.columns)):
            cell = tbl[i+1, j]
            val = table.iloc[i, j]
            if val > 0:
                alpha = min(val / 4, 1.0)
                cell.set_facecolor((0.46, 0.79, 0.60, alpha))
                cell.set_text_props(color='white', fontweight='bold')
            else:
                cell.set_facecolor('#2a2a2a')
                cell.set_text_props(color='#444444')
            cell.set_edgecolor('#444444')
        tbl[i+1, -1].set_facecolor('#1a3a2a')
        tbl[i+1, -1].set_text_props(color='#a8e6a3', fontweight='bold')
    
    for j in range(len(col_labels)):
        tbl[0, j].set_facecolor('#333333')
        tbl[0, j].set_text_props(color='#cccccc', fontweight='bold')
        tbl[0, j].set_edgecolor('#444444')
    
    ax.set_title(title, fontsize=12, color='#cccccc', fontweight='bold', pad=8)

plt.tight_layout()
plt.savefig('/tmp/egt_dual_tables.png', dpi=200, bbox_inches='tight', facecolor='#1a1a1a')
plt.savefig('/tmp/egt_dual_tables.pdf', bbox_inches='tight', facecolor='#1a1a1a')
plt.close()
print('Saved: /tmp/egt_dual_tables.png + .pdf')


Saved: /tmp/egt_dual_tables.png + .pdf


## Summary

- **Observational:** 20 studies, dominated by Cognitive/Neuro (8) and Nutrition/Dietary (6)
- **Pure EGT RCTs:** 3-4 completed — Yau 2024 (cognition, 1yr), Katsube 2022 (sleep, 4wk), Tian 2021 (metabolic, 12wk protocol), 120mg ovarian 2026 (open-label)
- **Gap:** No pure EGT RCT has measured epigenetic aging clocks — this is where our trial fits
- Mushroom-based RCTs excluded: coincident bioactives (beta-glucans, triterpenoids, vitamin D) cannot be attributed to EGT alone